# Ingesta bronze - order_products (Auto Loader)
Se usa Auto Loader con `trigger(availableNow=True)`, de modo que solo se procesan los archivos nuevos de la carpeta y luego el proceso termina.
El checkpoint guarda los archivos ya leidos, asi una nueva ejecucion no duplica datos.
En la carpeta estan `order_products__prior.csv` y `order_products__train.csv`; el origen se guarda en la columna `eval_set`.

In [ ]:
%run ./00_utils

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType

params = obtener_parametros()
CATALOGO, STORAGE = params["catalogo"], params["storage"]
ENTIDAD = "order_products"
TABLA_DESTINO = f"{CATALOGO}.bronze.{ENTIDAD}"
CHECKPOINT = ruta_tabla(STORAGE, "bronze", CATALOGO, f"_checkpoints/{ENTIDAD}")

In [ ]:
order_products_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("add_to_cart_order", IntegerType(), True),
    StructField("reordered", IntegerType(), True),
])

In [ ]:
try:
    ruta = ruta_raw(STORAGE, ENTIDAD)
    log(f"Auto Loader sobre {ruta}", entidad=ENTIDAD)

    df_stream = (spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", "csv")
                 .option("header", True)
                 .option("sep", ";")
                 .schema(order_products_schema)
                 .load(ruta))

    # Se obtiene prior o train a partir del nombre del archivo
    df_bronze = (agregar_auditoria(df_stream)
                 .withColumn("eval_set",
                             F.regexp_extract(F.col("source_file"), r"order_products__(\w+)\.csv", 1))
                 .select("order_id", "product_id", "add_to_cart_order", "reordered",
                         "eval_set", "ingestion_date", "source_file"))

    query = (df_bronze.writeStream
             .option("checkpointLocation", CHECKPOINT)
             .outputMode("append")
             .trigger(availableNow=True)
             .queryName(f"autoloader_{ENTIDAD}")
             .toTable(TABLA_DESTINO))
    query.awaitTermination()

    log(f"Ultimo batch: {query.lastProgress}", entidad=ENTIDAD)
    validar_no_vacio(spark.table(TABLA_DESTINO), TABLA_DESTINO)

except Exception as e:
    log(f"Error en la ingesta: {e}", "ERROR", ENTIDAD)
    raise

In [ ]:
display(spark.table(TABLA_DESTINO).groupBy("eval_set").count())